# Phase 6: Feature Steering

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Goal:** Establish causal relationships between features and model behavior, and demonstrate control of textual register.

**What this notebook does:**
1. Loads model, SAE, and selected features (from Phases 3-5)
2. Implements the **steering mechanism**: feature injection/suppression during the forward pass
3. **Ablation experiments**: suppresses PT-specific features (crase, clitics, gender) and measures the effect on generation
4. **Register amplification experiments**: amplifies textual-register features (legal, scientific, colloquial, journalistic) and evaluates tone control
5. **Quantitative evaluation**: measures changes in token distribution and coherence
6. Saves results and generates comparative visualizations (before/after)

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4)
- Checkpoints from the previous phases (`stats_*.pt`, `fase4_probes_results.pt`)
- Estimated runtime: ~30-45 min on T4

In [1]:
!pip install sae-lens transformer-lens -q
print()
print("Installation complete. Restart the runtime before continuing:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Then skip this cell and run from the next one.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.9/290.9 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 10
from tqdm.auto import tqdm
import time
import os
import json
import copy

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('Sem GPU detectada.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4 (15.6 GB)
Device: cuda
PyTorch: 2.10.0+cu128


In [2]:
from huggingface_hub import login
login()

In [3]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Carregando SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

Carregando Gemma 2 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Modelo: gemma-2-2b | Layers: 26 | d_model: 2304

Carregando SAE: layer_13/width_16k/canonical...


layer_13/width_16k/average_l0_84/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

SAE: 16384 features | d_in: 2304 | hook: blocks.13.hook_resid_post


/tmp/ipykernel_11645/3908203106.py:17: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./checkpoints/ for Colab)

FEATURES_PT = {
    "crase": 4584,
    "enclise": 2817,
    "proclise": 6215,
    "genero_feminino": 1215,
    "infinitivo_pessoal": 10349,
    "contracoes": 2294,
}

FEATURES_REGISTRO = {
    "coloquial": 1082,
    "cientifico": 5880,
    "juridico_1": 2294,
    "juridico_2": 12269,
    "jornalistico_1": 16057,
    "jornalistico_2": 10478,
    "coloquial_2": 15135,
}

print("PT-specific features for ablation:")
for name, fid in FEATURES_PT.items():
    print(f"  {name}: Feature {fid}")

print()
print("Register features for amplification:")
for name, fid in FEATURES_REGISTRO.items():
    print(f"  {name}: Feature {fid}")

Features PT-específicas para ablação:
  crase: Feature 4584
  enclise: Feature 2817
  proclise: Feature 6215
  genero_feminino: Feature 1215
  infinitivo_pessoal: Feature 10349
  contracoes: Feature 2294

Features de registro para amplificação:
  coloquial: Feature 1082
  cientifico: Feature 5880
  juridico_1: Feature 2294
  juridico_2: Feature 12269
  jornalistico_1: Feature 16057
  jornalistico_2: Feature 10478
  coloquial_2: Feature 15135


## 1. Feature Steering Mechanism

Steering works by intercepting the forward pass at the SAE layer (layer 13, residual stream):

1. The model processes tokens normally up to layer 13
2. We intercept the residual-stream activations
3. We pass them through the SAE encoder to obtain feature activations
4. We **modify** the target feature activations (zero for ablation, multiply for amplification)
5. We pass them through the SAE decoder to reconstruct the modified residual stream
6. The model continues the forward pass with the altered activations

**Important:** Steering is applied token by token during autoregressive generation.

In [ ]:
def generate_with_steering(model, sae, tokenizer, prompt, feature_ids,
                          multipliers, max_new_tokens=100, temperature=0.7,
                          hook_name=HOOK_NAME):
    """
    Generate text with feature steering applied at the SAE layer.

    Args:
        feature_ids: list of feature indices to steer
        multipliers: list of multipliers (0.0 = ablate, >1.0 = amplify, <0 = suppress)
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    feature_ids_t = torch.tensor(feature_ids, device=device)
    multipliers_t = torch.tensor(multipliers, dtype=torch.float16, device=device)

    def steering_hook(value, hook):
        with torch.no_grad():
            sae_input = value
            sae_acts = sae.encode(sae_input)
            modified_acts = sae_acts.clone()
            for fid, mult in zip(feature_ids_t, multipliers_t):
                modified_acts[:, :, fid] = modified_acts[:, :, fid] * mult
            reconstructed = sae.decode(modified_acts)
            error = sae_input - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model.run_with_hooks(
                generated,
                fwd_hooks=[(hook_name, steering_hook)],
            )
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    return result


def generate_baseline(model, tokenizer, prompt, max_new_tokens=100,
                      temperature=0.7):
    """Generate text without any steering (baseline)."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    return result


tokenizer = model.tokenizer
print("Mecanismo de steering implementado.")
print(f"Tokenizer: {tokenizer.__class__.__name__} | Vocab: {tokenizer.vocab_size}")

Mecanismo de steering implementado.
Tokenizer: GemmaTokenizerFast | Vocab: 256000


## 2. Ablation Experiments

We suppress PT-specific features (multiplier = 0.0) and observe the effect on generation.

**Hypotheses:**
- Suppressing the crase feature (4584) → reduces correct crase usage
- Suppressing the feminine gender feature (1215) → affects gender agreement
- Suppressing the enclisis feature (2817) → shifts clitic position

In [8]:
print("=" * 70)
print("EXPERIMENT 1: CRASE FEATURE ABLATION (Feature 4584)")
print("=" * 70)

prompts_crase = [
    "Eu vou à",
    "A proposta foi encaminhada à",
    "O aluno foi à escola e depois voltou à",
    "Refiro-me à questão levantada pela",
]

torch.manual_seed(42)
results_crase = []

for prompt in prompts_crase:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["crase"]],
                                     multipliers=[0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_crase.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": 4584,
        "operation": "ablation",
    })

print()
print("Crase ablation complete.")

EXPERIMENTO 1: ABLAÇÃO DA FEATURE DE CRASE (Feature 4584)

Prompt: "Eu vou à"
--------------------------------------------------
  Baseline:  Eu vou à minha casa e vejo uma mensagem no meu celular. É uma mensagem de um amigo meu. Ele está me dizendo que ele está com problemas de saúde. Ele está com problemas de saúde e ele precisa de ajuda. Eu não sei o que fazer. Eu
  Ablação:   Eu vou à loja de roupas da minha cidade e encontro uma peça que eu gosto. Eu vou ao caixa, pago e saio da loja. Mas, o que eu não sabia é que o vendedor estava me cobrando mais de R$ 100,

Prompt: "A proposta foi encaminhada à"
--------------------------------------------------
  Baseline:  A proposta foi encaminhada à Câmara de Vereadores de Aracaju, por meio da Comissão de Educação, Cultura e Turismo, e foi aprovada em primeira discussão.

O projeto de lei, que já está em análise na Comissão de Educação, Cultura e Turismo, prev
  Ablação:   A proposta foi encaminhada à Câmara Municipal de São Paulo pelo vere

In [9]:
print("=" * 70)
print("EXPERIMENT 2: FEMININE GENDER FEATURE ABLATION (Feature 1215)")
print("=" * 70)

prompts_genero = [
    "A menina bonita foi à",
    "A professora explicou que a aluna era muito",
    "Ela é uma pessoa muito",
    "A diretora da empresa apresentou a nova",
]

torch.manual_seed(42)
results_genero = []

for prompt in prompts_genero:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["genero_feminino"]],
                                     multipliers=[0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_genero.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": 1215,
        "operation": "ablation",
    })

print()
print("Feminine gender ablation complete.")

EXPERIMENTO 2: ABLAÇÃO DA FEATURE DE GÊNERO FEMININO (Feature 1215)

Prompt: "A menina bonita foi à"
--------------------------------------------------
  Baseline:  A menina bonita foi à escola e ao chegar lá, viu uma menina bonita e disse:

- Eu sou a mais bonita do mundo, mas você é a mais bonita do mundo.

A menina bonita disse:

- Você é a mais bonita do mundo, mas eu
  Ablação:   A menina bonita foi à feira e comprou um bolo.

A bolo é um alimento muito gostoso.

A bolo é um alimento muito gostoso.

A bolo é um alimento muito gostoso.

A bolo é um alimento muito gostoso.

A bolo

Prompt: "A professora explicou que a aluna era muito"
--------------------------------------------------
  Baseline:  A professora explicou que a aluna era muito tímida e que, quando ela estava com a mãe, não falava com ninguém.

“Ela não falava com ninguém. Ela não tinha amigos. Ela era muito tímida. Mas quando a mãe era lá, ela falava com todo
  Ablação:   A professora explicou que a aluna era muito tím

In [10]:
print("=" * 70)
print("EXPERIMENT 3: CLITIC FEATURES ABLATION (2817 + 6215)")
print("=" * 70)

prompts_cliticos = [
    "Diga-me o que",
    "Ele apresentou-se como",
    "Faça-me um favor e",
    "Entregou-lhe o documento e",
]

torch.manual_seed(42)
results_cliticos = []

for prompt in prompts_cliticos:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["enclise"],
                                                  FEATURES_PT["proclise"]],
                                     multipliers=[0.0, 0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_cliticos.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": [2817, 6215],
        "operation": "ablation",
    })

print()
print("Clitics ablation complete.")

EXPERIMENTO 3: ABLAÇÃO DAS FEATURES DE CLÍTICOS (2817 + 6215)

Prompt: "Diga-me o que"
--------------------------------------------------
  Baseline:  Diga-me o que você acha que é o melhor lugar para ir para o fim do mundo?

Eu acho que é a Ilha de Páscoa, na Austrália.

O que você acha?

O que você acha que é o melhor lugar para ir para
  Ablação:   Diga-me o que você acha do filme "O Diabo Veste Prada" e o que você acha que o filme tem a ver com a vida real.

O Diabo Veste Prada é um filme que mostra a vida de uma jovem aspirante a jornalista que

Prompt: "Ele apresentou-se como"
--------------------------------------------------
  Baseline:  Ele apresentou-se como um homem de 24 anos, com cabelo curto e olhos azuis. Ele estava usando uma camisa branca e uma calça preta. Ele também estava usando uma jaqueta preta. Ele estava usando um relógio de pulso. Ele estava usando uma jaqueta
  Ablação:   Ele apresentou-se como um homem de 25 anos, de 1,70m de altura, com cabelo loiro e olhos 

## 3. Register Amplification Experiments

We amplify textual-register features (multiplier > 1.0) to control the tone of generation.

**Hypotheses:**
- Amplifying the legal feature (2294, 12269) → text with a more formal/legal tone
- Amplifying the scientific feature (5880) → text with a more academic tone
- Amplifying the colloquial feature (1082) → text with a more informal tone
- Amplifying the journalistic feature (16057, 10478) → text with a more news-like tone

**Strategy:** We use the same neutral prompt and vary only the amplified register.

In [11]:
print("=" * 70)
print("EXPERIMENT 4: REGISTER CONTROL — SAME PROMPT, DIFFERENT REGISTERS")
print("=" * 70)

REGISTER_MULT = 8.0

prompts_registro = [
    "O governo anunciou novas medidas para",
    "A pesquisa sobre inteligência artificial mostra que",
    "O problema da educação no Brasil é que",
    "Os dados indicam que a economia brasileira",
]

register_configs = {
    "baseline": {"ids": [], "mults": []},
    "juridico": {
        "ids": [FEATURES_REGISTRO["juridico_1"], FEATURES_REGISTRO["juridico_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
    "cientifico": {
        "ids": [FEATURES_REGISTRO["cientifico"]],
        "mults": [REGISTER_MULT],
    },
    "coloquial": {
        "ids": [FEATURES_REGISTRO["coloquial"], FEATURES_REGISTRO["coloquial_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
    "jornalistico": {
        "ids": [FEATURES_REGISTRO["jornalistico_1"], FEATURES_REGISTRO["jornalistico_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
}

torch.manual_seed(42)
results_registro = []

for prompt in prompts_registro:
    print(f"\nPrompt: \"{prompt}\"")
    print("=" * 60)

    for reg_name, reg_cfg in register_configs.items():
        if reg_name == "baseline":
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=80, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=reg_cfg["ids"],
                                          multipliers=reg_cfg["mults"],
                                          max_new_tokens=80, temperature=0.3)
        label = reg_name.upper()
        print(f"  [{label:>13}] {text}")

        results_registro.append({
            "prompt": prompt,
            "register": reg_name,
            "text": text,
            "features": reg_cfg["ids"],
            "multiplier": REGISTER_MULT if reg_name != "baseline" else 0,
        })

    print()

print("Register control complete.")

EXPERIMENTO 4: CONTROLE DE REGISTRO — MESMO PROMPT, REGISTROS DIFERENTES

Prompt: "O governo anunciou novas medidas para"
  [     BASELINE] O governo anunciou novas medidas para o combate ao coronavírus, entre elas a suspensão de viagens internacionais e a proibição de eventos com mais de 50 pessoas. A decisão foi tomada no início da tarde desta segunda-feira (16).

O decreto, que ainda precisa ser publicado no Diário Oficial da União, também proíbe o uso de máscaras em lugares públicos, como aeroportos, rodoviárias,
  [     JURIDICO] O governo anunciou novas medidas para combater a crise da Covid-19. As medidas são mais restritivas e incluem o fechamento de 1500 estabelecimentos comerciais, como bares, restaurantes, shoppings e cinemas.

A medida foi anunciada pelo presidente Jair Bolsonaro (sem partido) em uma coletiva de 20h. O presidente também anunciou que o uso de máscaras será obrigatório em todos os
  [   CIENTIFICO] O governo anunciou novas medidas para conter a pandemia, que 

## 4. Multiplier Sweep

To understand the ideal steering intensity, we vary the multiplier from 0× to 16× for one register feature and evaluate the progressive effect.

In [12]:
print("=" * 70)
print("EXPERIMENT 5: MULTIPLIER SWEEP — LEGAL FEATURE (2294)")
print("=" * 70)

sweep_prompt = "O réu foi"
sweep_multipliers = [0.0, 1.0, 2.0, 4.0, 8.0, 12.0, 16.0]

torch.manual_seed(42)
results_sweep = []

for mult in sweep_multipliers:
    if mult == 1.0:
        text = generate_baseline(model, tokenizer, sweep_prompt,
                                 max_new_tokens=80, temperature=0.3)
        label = "1.0x (baseline)"
    elif mult == 0.0:
        text = generate_with_steering(model, sae, tokenizer, sweep_prompt,
                                      feature_ids=[FEATURES_REGISTRO["juridico_1"]],
                                      multipliers=[0.0],
                                      max_new_tokens=80, temperature=0.3)
        label = "0.0x (ablado)"
    else:
        text = generate_with_steering(model, sae, tokenizer, sweep_prompt,
                                      feature_ids=[FEATURES_REGISTRO["juridico_1"]],
                                      multipliers=[mult],
                                      max_new_tokens=80, temperature=0.3)
        label = f"{mult:.0f}x"

    print(f"  [{label:>15}] {text}")
    results_sweep.append({
        "multiplier": mult,
        "text": text,
        "feature": 2294,
    })

print()
print("Multiplier sweep complete.")

EXPERIMENTO 5: SWEEP DE MULTIPLICADORES — FEATURE JURÍDICA (2294)
  [  0.0x (ablado)] O réu foi condenado a 15 anos de prisão pelo homicídio de um policial militar e a 10 anos por tentativa de homicídio contra outro. Ele foi condenado a mais 15 anos de prisão por tentativa de homicídio contra um terceiro policial militar.

O crime ocorreu em 10 de fevereiro de 2017, quando o policial militar foi ao residência do réu para
  [1.0x (baseline)] O réu foi condenado a 18 anos e 10 meses de prisão, pelos crimes de homicídio qualificado, ocultação de cadáver e receptação. O crime ocorreu em 2017, em Ibirama.

<strong>LEIA TAMBÉM:</strong>

O crime ocorreu em 2017, quando o réu foi preso em flagrante por tráfico de drogas. Ele confessou o
  [             2x] O réu foi condenado a 15 anos e 10 meses de prisão pelo crime de homicídio qualificado, praticado contra a vítima, que foi encontrada morta em um matagal, em 2013.

O crime ocorreu em 2013, em uma residência no bairro Jardim América, em Ita

## 5. Quantitative Activation Analysis

For each experiment, we measure the SAE activations on the generated text and compare:
- Mean activation of the target features (baseline vs steered)
- Distribution of active features

In [15]:
def measure_feature_activations(model, sae, tokenizer, text, feature_ids,
                               hook_name=HOOK_NAME):
    """Measure SAE feature activations for a given text."""
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    activations = {}

    def capture_hook(value, hook):
        activations["resid"] = value.detach()
        return value

    with torch.no_grad():
        model.run_with_hooks(input_ids, fwd_hooks=[(hook_name, capture_hook)])

    resid = activations["resid"]
    sae_acts = sae.encode(resid)

    results = {}
    for fid in feature_ids:
        acts = sae_acts[0, :, fid].detach().cpu().numpy()
        results[fid] = {
            "mean": float(np.mean(acts)),
            "max": float(np.max(acts)),
            "nonzero_frac": float(np.mean(acts > 0)),
        }

    total_active = (sae_acts[0] > 0).sum(dim=-1).float()
    results["total_active_mean"] = float(total_active.mean().cpu())
    results["total_active_std"] = float(total_active.std().cpu())

    return results


print("Quantitative analysis: Crase Ablation")
print("-" * 50)

for r in results_crase[:2]:
    print(f"\nPrompt: \"{r['prompt']}\"")

    base_acts = measure_feature_activations(
        model, sae, tokenizer, r["baseline"], [4584])
    steer_acts = measure_feature_activations(
        model, sae, tokenizer, r["steered"], [4584])

    print(f"  Baseline  — Feature 4584: mean={base_acts[4584]['mean']:.3f}, "
          f"max={base_acts[4584]['max']:.3f}, "
          f"active={base_acts[4584]['nonzero_frac']*100:.0f}%")
    print(f"  Ablation  — Feature 4584: mean={steer_acts[4584]['mean']:.3f}, "
          f"max={steer_acts[4584]['max']:.3f}, "
          f"active={steer_acts[4584]['nonzero_frac']*100:.0f}%")
    print(f"  Features ativas (baseline): {base_acts['total_active_mean']:.0f} ± "
          f"{base_acts['total_active_std']:.0f}")
    print(f"  Active features (ablation):  {steer_acts['total_active_mean']:.0f} ± "
          f"{steer_acts['total_active_std']:.0f}")

print()
print("Quantitative analysis: Register Amplification")
print("-" * 50)

prompt_test = prompts_registro[0]
for reg_name in ["baseline", "juridico", "cientifico", "coloquial"]:
    matching = [r for r in results_registro
                if r["prompt"] == prompt_test and r["register"] == reg_name]
    if matching:
        r = matching[0]
        fids = list(FEATURES_REGISTRO.values())
        acts = measure_feature_activations(model, sae, tokenizer, r["text"], fids)
        print(f"\n  [{reg_name.upper():>13}]")
        for fname, fid in FEATURES_REGISTRO.items():
            if fid in acts:
                print(f"    Feature {fid} ({fname}): "
                      f"mean={acts[fid]['mean']:.2f}, max={acts[fid]['max']:.2f}")

Análise quantitativa: Ablação de Crase
--------------------------------------------------

Prompt: "Eu vou à"
  Baseline  — Feature 4584: mean=0.457, max=24.695, active=2%
  Ablação   — Feature 4584: mean=0.797, max=24.695, active=4%
  Features ativas (baseline): 188 ± 846
  Features ativas (ablação):  197 ± 845

Prompt: "A proposta foi encaminhada à"
  Baseline  — Feature 4584: mean=0.387, max=22.064, active=2%
  Ablação   — Feature 4584: mean=0.588, max=22.064, active=4%
  Features ativas (baseline): 189 ± 823
  Features ativas (ablação):  185 ± 824

Análise quantitativa: Amplificação de Registro
--------------------------------------------------

  [     BASELINE]
    Feature 1082 (coloquial): mean=0.05, max=4.33
    Feature 5880 (cientifico): mean=0.49, max=19.26
    Feature 2294 (juridico_1): mean=1.45, max=27.53
    Feature 12269 (juridico_2): mean=1.09, max=28.71
    Feature 16057 (jornalistico_1): mean=1.87, max=34.02
    Feature 10478 (jornalistico_2): mean=0.39, max=11.80
   

## 6. Visualizations

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

experiments = [
    ("Crase (F4584)", results_crase, [4584]),
    ("Gênero (F1215)", results_genero, [1215]),
    ("Clíticos (F2817+6215)", results_cliticos, [2817, 6215]),
]

for ax, (title, results, fids) in zip(axes, experiments):
    base_means = []
    steer_means = []

    for r in results:
        b_acts = measure_feature_activations(model, sae, tokenizer, r["baseline"], fids)
        s_acts = measure_feature_activations(model, sae, tokenizer, r["steered"], fids)

        base_val = np.mean([b_acts[f]["mean"] for f in fids])
        steer_val = np.mean([s_acts[f]["mean"] for f in fids])
        base_means.append(base_val)
        steer_means.append(steer_val)

    x = np.arange(len(results))
    width = 0.35
    ax.bar(x - width/2, base_means, width, label="Baseline", color="#2196F3")
    ax.bar(x + width/2, steer_means, width, label="Ablação (0x)", color="#F44336")
    ax.set_title(f"Ablação: {title}")
    ax.set_xlabel("Prompt")
    ax.set_ylabel("Ativação média")
    ax.set_xticks(x)
    ax.set_xticklabels([f"P{i+1}" for i in x])
    ax.legend()

plt.tight_layout()
plt.savefig("fig_fase6_ablation_comparison.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_fase6_ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_fase6_ablation_comparison.png")

Figura salva: fig_fase6_ablation_comparison.png


In [17]:
fig, axes = plt.subplots(1, len(prompts_registro), figsize=(5*len(prompts_registro), 5))
if len(prompts_registro) == 1:
    axes = [axes]

registers_order = ["juridico", "cientifico", "coloquial", "jornalistico"]
colors = {"juridico": "#8B0000", "cientifico": "#1565C0",
          "coloquial": "#FF9800", "jornalistico": "#2E7D32"}

for ax, prompt in zip(axes, prompts_registro):
    register_data = {}
    for reg in registers_order:
        matching = [r for r in results_registro
                    if r["prompt"] == prompt and r["register"] == reg]
        if matching:
            fids = list(FEATURES_REGISTRO.values())
            acts = measure_feature_activations(model, sae, tokenizer,
                                               matching[0]["text"], fids)
            register_data[reg] = {fname: acts[fid]["mean"]
                                  for fname, fid in FEATURES_REGISTRO.items()
                                  if fid in acts}

    feat_names = list(FEATURES_REGISTRO.keys())
    x = np.arange(len(feat_names))
    width = 0.2

    for i, reg in enumerate(registers_order):
        if reg in register_data:
            vals = [register_data[reg].get(fn, 0) for fn in feat_names]
            ax.bar(x + i*width, vals, width, label=reg.capitalize(),
                   color=colors[reg], alpha=0.85)

    short_prompt = prompt[:40] + "..."
    ax.set_title(short_prompt, fontsize=9)
    ax.set_xticks(x + 1.5*width)
    ax.set_xticklabels([f"F{FEATURES_REGISTRO[fn]}" for fn in feat_names],
                       rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Ativação média")
    ax.legend(fontsize=7)

plt.suptitle("Ativação de Features por Registro Amplificado", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_fase6_register_steering.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_fase6_register_steering.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_fase6_register_steering.png")

Figura salva: fig_fase6_register_steering.png


In [18]:
fig, ax = plt.subplots(figsize=(10, 5))

sweep_acts_mean = []
sweep_acts_max = []

for r in results_sweep:
    acts = measure_feature_activations(model, sae, tokenizer, r["text"], [2294])
    sweep_acts_mean.append(acts[2294]["mean"])
    sweep_acts_max.append(acts[2294]["max"])

ax.plot(sweep_multipliers, sweep_acts_mean, 'o-', label="Ativação média (F2294)",
        color="#8B0000", linewidth=2, markersize=8)
ax.plot(sweep_multipliers, sweep_acts_max, 's--', label="Ativação máxima (F2294)",
        color="#B71C1C", linewidth=1.5, markersize=6, alpha=0.7)

ax.set_xlabel("Multiplicador de Steering")
ax.set_ylabel("Ativação da Feature 2294")
ax.set_title("Sweep de Multiplicadores — Feature Jurídica (2294)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_fase6_multiplier_sweep.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_fase6_multiplier_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_fase6_multiplier_sweep.png")

Figura salva: fig_fase6_multiplier_sweep.png


## 7. Summary and Saving of Results

In [19]:
all_results = {
    "ablation": {
        "crase": results_crase,
        "genero": results_genero,
        "cliticos": results_cliticos,
    },
    "register_steering": results_registro,
    "multiplier_sweep": results_sweep,
    "config": {
        "features_pt": FEATURES_PT,
        "features_registro": FEATURES_REGISTRO,
        "register_multiplier": REGISTER_MULT,
        "sweep_multipliers": sweep_multipliers,
        "model": "gemma-2-2b",
        "sae": SAE_ID,
        "layer": LAYER,
    }
}

save_path = "fase6_steering_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f"Resultados salvos: {save_path}")

if os.path.exists(SAVE_DIR):
    drive_path = SAVE_DIR + "fase6_steering_results.json"
    with open(drive_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"Resultados salvos no Drive: {drive_path}")

print()
print("=" * 70)
print("PHASE 6 SUMMARY")
print("=" * 70)
print(f"Ablation experiments: 3 (crase, gender, clitics)")
print(f"  Crase prompts: {len(results_crase)}")
print(f"  Gender prompts: {len(results_genero)}")
print(f"  Clitics prompts: {len(results_cliticos)}")
print(f"Experimentos de registro: 4 registros × {len(prompts_registro)} prompts")
print(f"Sweep de multiplicadores: {len(sweep_multipliers)} valores")
print(f"Figuras geradas: 3")
print(f"  - fig_fase6_ablation_comparison.png")
print(f"  - fig_fase6_register_steering.png")
print(f"  - fig_fase6_multiplier_sweep.png")
print()

Resultados salvos: fase6_steering_results.json
Resultados salvos no Drive: ./checkpoints/fase6_steering_results.json

RESUMO DA FASE 6
Experimentos de ablação: 3 (crase, gênero, clíticos)
  Prompts de crase: 4
  Prompts de gênero: 4
  Prompts de clíticos: 4
Experimentos de registro: 4 registros × 4 prompts
Sweep de multiplicadores: 7 valores
Figuras geradas: 3
  - fig_fase6_ablation_comparison.png
  - fig_fase6_register_steering.png
  - fig_fase6_multiplier_sweep.png



In [ ]:
# placeholder